# Setup

In [ ]:
!pip install -q llama-index graspologic numpy==1.24.4 scipy==1.12.0

Ten kod instaluje kilka bibliotek w środowisku Pythona za pomocą narzędzia pip:

```
!pip install -q llama-index graspologic numpy==1.24.4 scipy==1.12.0
```

Znak wykrzyknika na początku oznacza, że to polecenie terminalowe wykonywane bezpośrednio z notatnika Jupyter. Flaga `-q` (quiet) sprawia, że instalacja przebiega w trybie cichym, z ograniczoną ilością wyświetlanych komunikatów.

Instalowane biblioteki to:
- llama-index - narzędzie do budowania aplikacji wykorzystujących duże modele językowe (LLM) do pracy z danymi
- graspologic - biblioteka do analizy i wizualizacji grafów/sieci
- numpy w dokładnej wersji 1.24.4 - podstawowa biblioteka do obliczeń numerycznych
- scipy w dokładnej wersji 1.12.0 - zaawansowana biblioteka do obliczeń naukowych i technicznych

Określenie konkretnych wersji numpy i scipy wynika z tego, że kod wymaga tych specyficznych wersji do poprawnego działania ze względu na kompatybilność z pozostałymi bibliotekami i wykorzystywane funkcje.

In [ ]:
# Standard library imports
import asyncio
import os
import re
from typing import Any, Callable, Dict, List, Optional, Union

# Third-party library imports
import nest_asyncio
import networkx as nx
import pandas as pd
from google.colab import userdata
from graspologic.partition import hierarchical_leiden
from IPython.display import Markdown, display

# LlamaIndex core imports
from llama_index.core import Document, PropertyGraphIndex
from llama_index.core.async_utils import run_jobs
from llama_index.core.bridge.pydantic import BaseModel, Field
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core.graph_stores.types import (
    EntityNode,
    KG_NODES_KEY,
    KG_RELATIONS_KEY,
    Relation,
)
from llama_index.core.indices.property_graph.utils import default_parse_triplets_fn
from llama_index.core.llms import ChatMessage, LLM
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.core.prompts.default_prompts import DEFAULT_KG_TRIPLET_EXTRACT_PROMPT
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.schema import BaseNode, TransformComponent

# LlamaIndex specific implementation imports
from llama_index.llms.openai import OpenAI

# Initialize nest_asyncio
nest_asyncio.apply()

asd

In [ ]:
class CFG:
    model = "gpt-4"
    embed = "text-embedding-3-small"
    chunk_size = 1024
    chunk_overlap = 20

Ta klasa `CFG` definiuje zestaw konfiguracji (stąd nazwa "CFG" - Configuration) używany prawdopodobnie w projekcie związanym z przetwarzaniem języka naturalnego lub uczeniem maszynowym. Jest to prosty kontener na wartości, który grupuje powiązane ustawienia w jednym miejscu.

Przeanalizujmy poszczególne atrybuty tej klasy:

- `model = "gpt-4"` - określa, który model językowy będzie używany w projekcie. W tym przypadku jest to GPT-4, zaawansowany model językowy stworzony przez OpenAI. To wskazuje, że projekt będzie korzystał z możliwości tego konkretnego modelu do generowania tekstu lub innych zadań przetwarzania języka.

- `embed = "text-embedding-3-small"` - definiuje, który model będzie wykorzystywany do tworzenia embedingów (wektorowych reprezentacji tekstu). "text-embedding-3-small" to również nazwa modelu od OpenAI, zaprojektowanego specjalnie do tworzenia oszczędnych, ale skutecznych reprezentacji wektorowych tekstu.

- `chunk_size = 1024` - określa wielkość fragmentów, na które będzie dzielony tekst podczas przetwarzania. Wartość 1024 prawdopodobnie oznacza liczbę tokenów lub znaków w jednym fragmencie. Dzielenie tekstu na fragmenty jest powszechną praktyką przy pracy z modelami językowymi, ponieważ pozwala na przetwarzanie długich tekstów, które mogłyby przekraczać maksymalną długość kontekstu modelu.

- `chunk_overlap = 20` - definiuje, ile jednostek (tokenów lub znaków) będzie się pokrywać między sąsiadującymi fragmentami tekstu. To nakładanie się fragmentów zapewnia ciągłość kontekstu podczas przetwarzania podzielonego tekstu i pomaga zapobiec utracie informacji na granicach fragmentów.


In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')


Ta linia kodu ustawia zmienną środowiskową `OPENAI_API_KEY` używając wartości pobranej z obiektu `userdata`. Zmienne środowiskowe to zmienne dostępne dla wszystkich procesów uruchomionych w danym środowisku, a w kontekście programowania często służą do przechowywania wrażliwych danych, takich jak klucze API.

Rozbijmy to na części:

1. `os.environ` to słownik Pythona, który reprezentuje zmienne środowiskowe systemu operacyjnego. Dostęp do niego wymaga zaimportowania modułu `os`.

2. `["OPENAI_API_KEY"]` wskazuje konkretną zmienną środowiskową, którą chcemy ustawić. Ta zmienna jest używana przez biblioteki OpenAI do uwierzytelniania żądań API.

3. `userdata.get('openaivision')` pobiera wartość klucza API z obiektu `userdata`, prawdopodobnie będącego bezpiecznym magazynem na dane użytkownika. Funkcja `get()` zazwyczaj służy do bezpiecznego odczytywania wartości ze słownika, zwracając `None` zamiast błędu, jeśli klucz nie istnieje.

Warto zauważyć ciekawy szczegół: wartość jest pobierana z klucza `'openaivision'`, co sugeruje, że może to być specyficzny klucz API używany do dostępu do usług wizyjnych OpenAI (np. modeli, które mogą analizować obrazy), a nie standardowy klucz OpenAI API.

Taka metoda przechowywania kluczy API jest zgodna z dobrymi praktykami bezpieczeństwa, ponieważ:
- Unika twardego kodowania kluczy API bezpośrednio w kodzie źródłowym
- Centralizuje zarządzanie danymi uwierzytelniającymi
- Prawdopodobnie korzysta z bezpiecznego mechanizmu przechowywania, takiego jak szyfrowane magazyny danych lub usługi zarządzania sekretami


# Dane

In [ ]:

news = pd.read_csv( "https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/news_articles.csv")[:50]

news.head()

,title,date,text
0,Chevron: Best Of Breed,2031-04-06T01:36:32.000000000+00:00,JHVEPhoto Like many companies in the O&G secto...
1,FirstEnergy (NYSE:FE) Posts Earnings Results,2030-04-29T06:55:28.000000000+00:00,FirstEnergy (NYSE:FE – Get Rating) posted its ...
2,Dáil almost suspended after Sinn Féin TD put p...,2023-06-15T14:32:11.000000000+00:00,The Dáil was almost suspended on Thursday afte...
3,Epic’s latest tool can animate hyperrealistic ...,2023-06-15T14:00:00.000000000+00:00,"Today, Epic is releasing a new tool designed t..."
4,"EU to Ban Huawei, ZTE from Internal Commission...",2023-06-15T13:50:00.000000000+00:00,The European Commission is planning to ban equ...



Te dwie linie kodu pobierają dane z pliku CSV i wyświetlają pierwsze kilka wierszy tych danych.

W pierwszej linii:
```
news = pd.read_csv("https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/news_articles.csv")[:50]
```

Dzieje się kilka rzeczy:
1. `pd.read_csv()` to funkcja z biblioteki pandas (skróconej jako `pd`), która wczytuje dane z pliku CSV do struktury danych zwanej DataFrame. DataFrame to tabela podobna do arkusza kalkulacyjnego, z wierszami i kolumnami.

2. URL `"https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/news_articles.csv"` wskazuje na plik CSV przechowywany w repozytorium GitHub użytkownika 'tomasonjo', w folderze 'blog-datasets'. Plik zawiera zbiór artykułów informacyjnych.

3. `[:50]` to tzw. "slicing" - metoda, która ogranicza liczbę wierszy do pierwszych 50. Oznacza to, że nawet jeśli oryginalny plik zawiera setki czy tysiące artykułów, kod wczyta tylko 50 pierwszych.

4. Wynik tej operacji jest przypisany do zmiennej `news`, która teraz zawiera DataFrame z 50 pierwszymi artykułami.

W drugiej linii:
```
news.head()
```

Funkcja `head()` wyświetla kilka pierwszych wierszy DataFrame (domyślnie 5). Jest to powszechna praktyka w analizie danych, pozwalająca szybko sprawdzić strukturę danych i ich zawartość bez wyświetlania całego zbioru, który może być bardzo duży.

Dzięki tym dwóm linikom mamy dostęp do próbki danych z artykułami informacyjnymi i możemy zobaczyć ich format. W typowym DataFrame z artykułami możemy spodziewać się takich kolumn jak tytuł, treść, data publikacji, autor, a być może także kategoria lub słowa kluczowe. Wynik `head()` wyświetli te dane w formie tabeli, co pozwala szybko ocenić, jakie informacje są dostępne w zbiorze danych i jak są one zorganizowane.

In [ ]:
documents = [ Document(text=f"{row['title']}: {row['text']}")  for i, row in news.iterrows() ]

Ta linia kodu tworzy listę obiektów `Document` na podstawie danych z DataFrame `news`. To kluczowy krok w przygotowaniu danych tekstowych do dalszego przetwarzania, prawdopodobnie w kontekście wektoryzacji tekstu lub analizy przy użyciu modeli językowych.

Rozłóżmy tę linię na części:

1. `documents = [...]` - tworzymy nową listę i przypisujemy ją do zmiennej `documents`.

2. `for i, row in news.iterrows()` - iterujemy przez wiersze DataFrame `news`. Metoda `iterrows()` zwraca pary (indeks, wiersz), gdzie `i` to indeks wiersza, a `row` to obiekt Series zawierający dane z tego wiersza.

3. `Document(text=f"{row['title']}: {row['text']}")` - dla każdego wiersza tworzymy nowy obiekt klasy `Document`. Jako argument `text` przekazujemy połączenie tytułu artykułu i jego treści, oddzielone dwukropkiem.

4. Format `f"{row['title']}: {row['text']}"` to tzw. f-string w Pythonie, który pozwala na wstawianie wartości zmiennych bezpośrednio do tekstu. W tym przypadku pobieramy wartości z kolumn 'title' i 'text' z bieżącego wiersza.

Klasa `Document` nie jest standardową klasą w Pythonie - pochodzi z biblioteki LlamaIndex lub podobnej. Obiekty tej klasy  służą jako kontenery na tekst, zachowujące również metadane lub umożliwiające operacje specyficzne dla przetwarzania dokumentów.

Efektem tej operacji jest lista 50 obiektów `Document`, każdy zawierający połączony tytuł i treść jednego artykułu. Taka struktura danych jest często używana w systemach przetwarzania języka naturalnego, gdy chcemy podzielić duży zbiór tekstów na mniejsze, łatwiejsze do zarządzania jednostki, zachowując jednocześnie informacje o źródle każdego fragmentu.


# LLM

In [ ]:
llm = OpenAI(model = CFG.model)

Ta linia kodu inicjalizuje obiekt `llm`, który będzie służył jako interfejs do modelu językowego GPT-4 OpenAI.

Analizując szczegółowo:

`OpenAI(model = CFG.model)` tworzy nowy obiekt klasy `OpenAI`. Ta klasa  pochodzi z LlamaIndex.

Parametr `model = CFG.model` określa, który konkretny model językowy OpenAI będzie używany. Wartość ta jest pobierana z wcześniej zdefiniowanej klasy konfiguracyjnej `CFG`. Jak pamiętamy z poprzedniego kodu, `CFG.model` ma wartość "gpt-4", co oznacza, że będziemy korzystać z modelu GPT-4.

Ta jedna linia ma duże znaczenie w całym systemie. Tworzy centralny komponent, który będzie odpowiedzialny za generowanie tekstu, odpowiadanie na pytania lub analizowanie tekstu - w zależności od tego, jak zostanie wykorzystany w dalszej części programu.

# GraphRAG setup

## Extraction

In [ ]:
class GraphRAGExtractor(TransformComponent):
    """Extract triples from a graph.

    Uses an LLM and a simple prompt + output parsing to extract paths (i.e. triples) and entity, relation descriptions from text.

    Args:
        llm (LLM):
            The language model to use.
        extract_prompt (Union[str, PromptTemplate]):
            The prompt to use for extracting triples.
        parse_fn (callable):
            A function to parse the output of the language model.
        num_workers (int):
            The number of workers to use for parallel processing.
        max_paths_per_chunk (int):
            The maximum number of paths to extract per chunk.
    """

    llm: LLM
    extract_prompt: PromptTemplate
    parse_fn: Callable
    num_workers: int
    max_paths_per_chunk: int

    def __init__(
        self,
        llm: Optional[LLM] = None,
        extract_prompt: Optional[Union[str, PromptTemplate]] = None,
        parse_fn: Callable = default_parse_triplets_fn,
        max_paths_per_chunk: int = 10,
        num_workers: int = 4,
    ) -> None:
        """Init params."""
        from llama_index.core import Settings

        if isinstance(extract_prompt, str):
            extract_prompt = PromptTemplate(extract_prompt)

        super().__init__(
            llm=llm or Settings.llm,
            extract_prompt=extract_prompt or DEFAULT_KG_TRIPLET_EXTRACT_PROMPT,
            parse_fn=parse_fn,
            num_workers=num_workers,
            max_paths_per_chunk=max_paths_per_chunk,
        )

    @classmethod
    def class_name(cls) -> str:
        return "GraphExtractor"

    def __call__(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """Extract triples from nodes."""
        return asyncio.run(
            self.acall(nodes, show_progress=show_progress, **kwargs)
        )

    async def _aextract(self, node: BaseNode) -> BaseNode:
        """Extract triples from a node."""
        assert hasattr(node, "text")

        text = node.get_content(metadata_mode="llm")
        try:
            llm_response = await self.llm.apredict(
                self.extract_prompt,
                text=text,
                max_knowledge_triplets=self.max_paths_per_chunk,
            )
            entities, entities_relationship = self.parse_fn(llm_response)
        except ValueError:
            entities = []
            entities_relationship = []

        existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
        existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
        metadata = node.metadata.copy()
        for entity, entity_type, description in entities:
            metadata[
                "entity_description"
            ] = description  # Not used in the current implementation. But will be useful in future work.
            entity_node = EntityNode(
                name=entity, label=entity_type, properties=metadata
            )
            existing_nodes.append(entity_node)

        metadata = node.metadata.copy()
        for triple in entities_relationship:
            subj, rel, obj, description = triple
            subj_node = EntityNode(name=subj, properties=metadata)
            obj_node = EntityNode(name=obj, properties=metadata)
            metadata["relationship_description"] = description
            rel_node = Relation(
                label=rel,
                source_id=subj_node.id,
                target_id=obj_node.id,
                properties=metadata,
            )

            existing_nodes.extend([subj_node, obj_node])
            existing_relations.append(rel_node)

        node.metadata[KG_NODES_KEY] = existing_nodes
        node.metadata[KG_RELATIONS_KEY] = existing_relations
        return node

    async def acall(
        self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
    ) -> List[BaseNode]:
        """Extract triples from nodes async."""
        jobs = []
        for node in nodes:
            jobs.append(self._aextract(node))

        return await run_jobs(
            jobs,
            workers=self.num_workers,
            show_progress=show_progress,
            desc="Extracting paths from text",
        )

Przeprowadźmy szczegółową analizę przedstawionego kodu, który implementuje klasę `GraphRAGExtractor` służącą do ekstrakcji trójek (tripletów) z grafów wiedzy.

Ta klasa jest częścią systemu RAG (Retrieval-Augmented Generation), który łączy wyszukiwanie informacji z generowaniem tekstu za pomocą modeli językowych. Klasa `GraphRAGExtractor` dziedziczy po `TransformComponent` i służy do przekształcania tekstu w strukturę grafu wiedzy.

Pierwszy fragment definiuje klasę i jej atrybuty:

```python
class GraphRAGExtractor(TransformComponent):
    """Extract triples from a graph...
    """

    llm: LLM
    extract_prompt: PromptTemplate
    parse_fn: Callable
    num_workers: int
    max_paths_per_chunk: int
```

Atrybuty klasy to:
- `llm` - model językowy używany do ekstrakcji informacji
- `extract_prompt` - szablon zapytania używany do ekstrakcji trójek
- `parse_fn` - funkcja przetwarzająca wynik zwrócony przez model
- `num_workers` - liczba równoległych procesów do przetwarzania
- `max_paths_per_chunk` - maksymalna liczba ścieżek do ekstrakcji z jednego fragmentu tekstu

Konstruktor klasy inicjalizuje te atrybuty:

```python
def __init__(
    self,
    llm: Optional[LLM] = None,
    extract_prompt: Optional[Union[str, PromptTemplate]] = None,
    parse_fn: Callable = default_parse_triplets_fn,
    max_paths_per_chunk: int = 10,
    num_workers: int = 4,
) -> None:
```

Warto zauważyć, że jeśli nie podamy modelu językowego (`llm`), zostanie użyty domyślny model z `Settings.llm`. Podobnie, jeśli nie podamy `extract_prompt`, zostanie użyty domyślny szablon `DEFAULT_KG_TRIPLET_EXTRACT_PROMPT`. Konstruktor zamienia również prosty string na obiekt `PromptTemplate`, jeśli taki zostanie podany.

Metoda `class_name` jest prosta - zwraca nazwę klasy jako string:

```python
@classmethod
def class_name(cls) -> str:
    return "GraphExtractor"
```

Metoda `__call__` jest przeciążeniem operatora wywołania funkcji, co pozwala na używanie obiektu jak funkcji:

```python
def __call__(
    self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
) -> List[BaseNode]:
    """Extract triples from nodes."""
    return asyncio.run(
        self.acall(nodes, show_progress=show_progress, **kwargs)
    )
```

Ta metoda po prostu uruchamia asynchroniczny odpowiednik `acall` za pomocą `asyncio.run()`.

Najważniejsza logika znajduje się w metodzie `_aextract`:

```python
async def _aextract(self, node: BaseNode) -> BaseNode:
    """Extract triples from a node."""
    assert hasattr(node, "text")

    text = node.get_content(metadata_mode="llm")
    try:
        llm_response = await self.llm.apredict(
            self.extract_prompt,
            text=text,
            max_knowledge_triplets=self.max_paths_per_chunk,
        )
        entities, entities_relationship = self.parse_fn(llm_response)
    except ValueError:
        entities = []
        entities_relationship = []
```

Ta metoda:
1. Sprawdza czy node ma atrybut `text`
2. Pobiera treść z node przy użyciu `node.get_content()`
3. Wysyła zapytanie do modelu językowego za pomocą `self.llm.apredict()`
4. Przetwarza odpowiedź przy użyciu `self.parse_fn`
5. W przypadku błędu, inicjalizuje puste listy

Następnie metoda aktualizuje metadane node'a:

```python
existing_nodes = node.metadata.pop(KG_NODES_KEY, [])
existing_relations = node.metadata.pop(KG_RELATIONS_KEY, [])
metadata = node.metadata.copy()
for entity, entity_type, description in entities:
    metadata[
        "entity_description"
    ] = description
    entity_node = EntityNode(
        name=entity, label=entity_type, properties=metadata
    )
    existing_nodes.append(entity_node)
```

Ten fragment pobiera istniejące węzły i relacje z metadanych, a następnie tworzy nowe węzły typu `EntityNode` dla każdej encji wykrytej przez model.

Kolejny fragment tworzy relacje między encjami:

```python
metadata = node.metadata.copy()
for triple in entities_relationship:
    subj, rel, obj, description = triple
    subj_node = EntityNode(name=subj, properties=metadata)
    obj_node = EntityNode(name=obj, properties=metadata)
    metadata["relationship_description"] = description
    rel_node = Relation(
        label=rel,
        source_id=subj_node.id,
        target_id=obj_node.id,
        properties=metadata,
    )

    existing_nodes.extend([subj_node, obj_node])
    existing_relations.append(rel_node)
```

Dla każdej trójki (podmiot, relacja, obiekt) tworzone są odpowiednie węzły i relacje, które dodawane są do list.

Na końcu metody, zaktualizowane listy są zapisywane z powrotem do metadanych node'a:

```python
node.metadata[KG_NODES_KEY] = existing_nodes
node.metadata[KG_RELATIONS_KEY] = existing_relations
return node
```

Ostatnia metoda `acall` implementuje asynchroniczne przetwarzanie wielu node'ów równolegle:

```python
async def acall(
    self, nodes: List[BaseNode], show_progress: bool = False, **kwargs: Any
) -> List[BaseNode]:
    """Extract triples from nodes async."""
    jobs = []
    for node in nodes:
        jobs.append(self._aextract(node))

    return await run_jobs(
        jobs,
        workers=self.num_workers,
        show_progress=show_progress,
        desc="Extracting paths from text",
    )
```

Ta metoda tworzy listę zadań, dodając wywołanie `_aextract` dla każdego node'a, a następnie uruchamia te zadania równolegle za pomocą funkcji pomocniczej `run_jobs`.

Podsumowując, `GraphRAGExtractor` to klasa, która:
1. Pobiera tekst z węzłów grafu
2. Wykorzystuje model językowy do identyfikacji encji i relacji w tekście
3. Tworzy węzły i krawędzie grafu wiedzy na podstawie wyekstrahowanych informacji
4. Aktualizuje metadane oryginalnych węzłów o te nowe informacje

Ta funkcjonalność jest kluczowa w systemach RAG, gdzie struktury grafowe pozwalają na lepsze reprezentowanie relacji między informacjami, co przekłada się na lepsze wyszukiwanie i generowanie odpowiedzi.

## Communities

In [ ]:
class GraphRAGStore(SimplePropertyGraphStore):
    community_summary = {}
    max_cluster_size = 5

    def generate_community_summary(self, text):
        """Generate summary for a given text using an LLM."""
        messages = [
            ChatMessage(
                role="system",
                content=(
                    "You are provided with a set of relationships from a knowledge graph, each represented as "
                    "entity1->entity2->relation->relationship_description. Your task is to create a summary of these "
                    "relationships. The summary should include the names of the entities involved and a concise synthesis "
                    "of the relationship descriptions. The goal is to capture the most critical and relevant details that "
                    "highlight the nature and significance of each relationship. Ensure that the summary is coherent and "
                    "integrates the information in a way that emphasizes the key aspects of the relationships."
                ),
            ),
            ChatMessage(role="user", content=text),
        ]
        response = OpenAI().chat(messages)
        clean_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return clean_response

    def build_communities(self):
        """Builds communities from the graph and summarizes them."""
        nx_graph = self._create_nx_graph()
        community_hierarchical_clusters = hierarchical_leiden(
            nx_graph, max_cluster_size=self.max_cluster_size
        )
        community_info = self._collect_community_info(
            nx_graph, community_hierarchical_clusters
        )
        self._summarize_communities(community_info)

    def _create_nx_graph(self):
        """Converts internal graph representation to NetworkX graph."""
        nx_graph = nx.Graph()
        for node in self.graph.nodes.values():
            nx_graph.add_node(str(node))
        for relation in self.graph.relations.values():
            nx_graph.add_edge(
                relation.source_id,
                relation.target_id,
                relationship=relation.label,
                description=relation.properties["relationship_description"],
            )
        return nx_graph

    def _collect_community_info(self, nx_graph, clusters):
        """Collect detailed information for each node based on their community."""
        community_mapping = {item.node: item.cluster for item in clusters}
        community_info = {}
        for item in clusters:
            cluster_id = item.cluster
            node = item.node
            if cluster_id not in community_info:
                community_info[cluster_id] = []

            for neighbor in nx_graph.neighbors(node):
                if community_mapping[neighbor] == cluster_id:
                    edge_data = nx_graph.get_edge_data(node, neighbor)
                    if edge_data:
                        detail = f"{node} -> {neighbor} -> {edge_data['relationship']} -> {edge_data['description']}"
                        community_info[cluster_id].append(detail)
        return community_info

    def _summarize_communities(self, community_info):
        """Generate and store summaries for each community."""
        for community_id, details in community_info.items():
            details_text = (
                "\n".join(details) + "."
            )  # Ensure it ends with a period
            self.community_summary[
                community_id
            ] = self.generate_community_summary(details_text)

    def get_community_summaries(self):
        """Returns the community summaries, building them if not already done."""
        if not self.community_summary:
            self.build_communities()
        return self.community_summary

Ten kod definiuje klasę `GraphRAGStore`, która rozszerza `SimplePropertyGraphStore`. Klasa ta zarządza grafem wiedzy i wykorzystuje algorytmy grupowania do identyfikacji społeczności (communities) w grafie, a następnie generuje dla nich podsumowania przy użyciu modelu językowego.

Kluczowe atrybuty klasy to:
- `community_summary` - słownik przechowujący podsumowania dla każdej społeczności
- `max_cluster_size` - maksymalny rozmiar klastra w algorytmie grupowania

Metoda `generate_community_summary` wykorzystuje model językowy do generowania podsumowań. Przyjmuje tekst zawierający relacje z grafu wiedzy i tworzy zapytanie do modelu OpenAI. Zapytanie składa się z dwóch wiadomości: instrukcji systemowej i treści od użytkownika. Model otrzymuje zadanie stworzenia zwięzłego podsumowania relacji między encjami, uwzględniając ich nazwy i opis relacji. Po otrzymaniu odpowiedzi, metoda czyści ją za pomocą wyrażenia regularnego, usuwając prefiks "assistant:", i zwraca jako wynik.

Metoda `build_communities` jest głównym procesem budowania społeczności. Składa się z trzech kroków:
1. Konwersja wewnętrznej reprezentacji grafu na graf NetworkX
2. Identyfikacja klastrów hierarchicznych za pomocą algorytmu Leiden
3. Zbieranie informacji o społecznościach i generowanie ich podsumowań

Metoda `_create_nx_graph` konwertuje wewnętrzną reprezentację grafu na format kompatybilny z biblioteką NetworkX. Dla każdego węzła w grafie źródłowym dodaje odpowiadający mu węzeł w grafie NetworkX, a dla każdej relacji dodaje krawędź z etykietą relacji i jej opisem.

Metoda `_collect_community_info` zbiera szczegółowe informacje o każdej społeczności. Tworzy słownik mapujący każdy węzeł na jego społeczność, a następnie dla każdego klastra zbiera informacje o relacjach między węzłami należącymi do tego samego klastra. Dla każdej takiej relacji tworzy tekstowy opis w formacie "węzeł1 -> węzeł2 -> relacja -> opis_relacji".

Metoda `_summarize_communities` generuje podsumowania dla każdej społeczności. Dla każdej społeczności łączy zebrane wcześniej opisy relacji w jeden tekst, a następnie wywołuje metodę `generate_community_summary` w celu wygenerowania zwięzłego podsumowania za pomocą modelu językowego.

Metoda `get_community_summaries` zwraca słownik podsumowań społeczności. Jeśli podsumowania nie zostały jeszcze wygenerowane, metoda najpierw wywołuje `build_communities`.

Ten kod implementuje zaawansowane podejście do analizy grafów wiedzy, łącząc algorytmy wykrywania społeczności (jak Leiden) z modelami językowymi do generowania podsumowań. Takie podejście jest szczególnie przydatne w systemach RAG (Retrieval-Augmented Generation), gdzie struktury grafowe mogą reprezentować złożone relacje między informacjami, a podsumowania społeczności mogą pomóc w lepszym zrozumieniu głównych tematów i związków w danych.

## Query

In [ ]:
class GraphRAGQueryEngine(CustomQueryEngine):
    graph_store: GraphRAGStore
    llm: LLM

    def custom_query(self, query_str: str) -> str:
        """Process all community summaries to generate answers to a specific query."""
        community_summaries = self.graph_store.get_community_summaries()
        community_answers = [
            self.generate_answer_from_summary(community_summary, query_str)
            for _, community_summary in community_summaries.items()
        ]

        final_answer = self.aggregate_answers(community_answers)
        return final_answer

    def generate_answer_from_summary(self, community_summary, query):
        """Generate an answer from a community summary based on a given query using LLM."""
        prompt = (
            f"Given the community summary: {community_summary}, "
            f"how would you answer the following query? Query: {query}"
        )
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content="I need an answer based on the above information.",
            ),
        ]
        response = self.llm.chat(messages)
        cleaned_response = re.sub(r"^assistant:\s*", "", str(response)).strip()
        return cleaned_response

    def aggregate_answers(self, community_answers):
        """Aggregate individual community answers into a final, coherent response."""
        # intermediate_text = " ".join(community_answers)
        prompt = "Combine the following intermediate answers into a final, concise response."
        messages = [
            ChatMessage(role="system", content=prompt),
            ChatMessage(
                role="user",
                content=f"Intermediate answers: {community_answers}",
            ),
        ]
        final_response = self.llm.chat(messages)
        cleaned_final_response = re.sub(
            r"^assistant:\s*", "", str(final_response)
        ).strip()
        return cleaned_final_response

Klasa `GraphRAGQueryEngine` to zaawansowany mechanizm odpowiadania na zapytania, który wykorzystuje strukturę grafową wiedzy i podsumowania społeczności do generowania kompleksowych odpowiedzi. Zobaczmy, jak ta klasa działa w szczegółach.

Klasa dziedziczy po `CustomQueryEngine` i posiada dwa główne komponenty:
- `graph_store` - instancja `GraphRAGStore`, która zarządza grafem wiedzy i podsumowaniami społeczności
- `llm` - model językowy używany do generowania odpowiedzi

Cały proces odpowiadania na zapytania opiera się na trzech kluczowych metodach:

## Metoda `custom_query`

Ta metoda to główny punkt wejścia dla zapytań. Przyjmuje zapytanie w formie tekstowej i zwraca na nie odpowiedź. Proces składa się z następujących kroków:

1. Pobiera wszystkie podsumowania społeczności z `graph_store`
2. Dla każdego podsumowania generuje częściową odpowiedź na zapytanie
3. Łączy wszystkie częściowe odpowiedzi w jedną, spójną wypowiedź

Podejście to jest podobne do metody "podziel i zwyciężaj" - zamiast próbować analizować cały graf wiedzy naraz, system dzieli go na mniejsze, sensowne fragmenty (społeczności), analizuje każdy fragment osobno, a następnie łączy wyniki.

## Metoda `generate_answer_from_summary`

Ta metoda wykorzystuje model językowy do wygenerowania odpowiedzi na podstawie pojedynczego podsumowania społeczności. Działa następująco:

1. Tworzy zapytanie łączące podsumowanie społeczności z oryginalnym pytaniem użytkownika
2. Konstruuje wiadomości dla modelu, składające się z instrukcji systemowej i prośby użytkownika
3. Wywołuje model językowy za pomocą metody `chat`
4. Oczyszcza otrzymaną odpowiedź, usuwając niepotrzebne elementy formatowania

Co ciekawe, zapytanie jest skonstruowane w taki sposób, że model otrzymuje kontekst zarówno z grafu wiedzy (w formie podsumowania społeczności), jak i oryginalnego pytania. To pozwala mu generować odpowiedzi, które są zarówno trafne (oparte na faktach z grafu wiedzy), jak i odpowiednie (odnoszące się konkretnie do pytania użytkownika).

## Metoda `aggregate_answers`

Ta metoda jest odpowiedzialna za połączenie wszystkich częściowych odpowiedzi w jedną, spójną wypowiedź. Proces wygląda następująco:

1. Tworzy instrukcję dla modelu językowego, aby połączył częściowe odpowiedzi
2. Konstruuje wiadomości dla modelu, przekazując wszystkie częściowe odpowiedzi jako "odpowiedzi pośrednie"
3. Wywołuje model językowy
4. Oczyszcza otrzymaną odpowiedź z niepotrzebnych elementów formatowania

Warto zauważyć, że w kodzie jest zakomentowana linia `# intermediate_text = " ".join(community_answers)`, co sugeruje, że wcześniej próbowano prostszego podejścia polegającego na mechanicznym łączeniu odpowiedzi. Obecne rozwiązanie wykorzystuje inteligencję modelu językowego do stworzenia spójnej syntezy, zamiast prostego połączenia tekstu.

## Dlaczego to podejście jest skuteczne?

To podejście wykorzystuje silne strony zarówno grafów, jak i modeli językowych:

1. **Grafy wiedzy** doskonale reprezentują relacje między encjami, pozwalając na identyfikację powiązanych informacji i grupowanie ich w sensowne społeczności.

2. **Algorytmy grupowania** (jak Leiden użyty w `GraphRAGStore`) identyfikują naturalne skupiska w danych, dzieląc złożony graf na mniejsze, bardziej spójne tematycznie fragmenty.

3. **Modele językowe** (LLM) są świetne w rozumieniu kontekstu i generowaniu spójnego tekstu, ale mogą mieć problem z przetwarzaniem zbyt dużej ilości informacji naraz. Dzięki podziałowi na społeczności, każde wywołanie modelu otrzymuje mniejszy, bardziej skoncentrowany zestaw danych.

4. **Podejście wieloetapowe** (generowanie odpowiedzi częściowych, a następnie ich agregacja) pozwala na lepsze wykorzystanie modelu językowego, dając mu możliwość skupienia się najpierw na szczegółach, a potem na szerszym obrazie.

Ta klasa stanowi elegancki przykład integracji strukturalnego podejścia do wiedzy (grafy) z generatywnymi możliwościami nowoczesnych modeli językowych, tworząc system, który może odpowiadać na złożone pytania w sposób zarówno precyzyjny, jak i naturalny dla człowieka.

# Pipeline

In [ ]:

splitter = SentenceSplitter( chunk_size = CFG.chunk_size,
    chunk_overlap = CFG.chunk_overlap,
)
nodes = splitter.get_nodes_from_documents(documents)

Kod ten tworzy instancję klasy `SentenceSplitter` i wykorzystuje ją do podziału dokumentów na mniejsze fragmenty (nodes). Przyjrzyjmy się dokładnie, co się tu dzieje i dlaczego jest to ważny krok w przetwarzaniu tekstu.

### Co robi ten kod?

Dzieli się na dwie główne operacje:

1. **Tworzenie obiektu dzielącego tekst**:
   ```python
   splitter = SentenceSplitter(
       chunk_size = CFG.chunk_size,
       chunk_overlap = CFG.chunk_overlap,
   )
   ```

2. **Zastosowanie podzielnika do dokumentów**:
   ```python
   nodes = splitter.get_nodes_from_documents(documents)
   ```

### Dlaczego dzielimy dokumenty?

Podział dokumentów na mniejsze fragmenty jest kluczowym elementem w systemach przetwarzania języka naturalnego, szczególnie w architekturach RAG (Retrieval-Augmented Generation), z kilku powodów:

1. **Ograniczenia modeli językowych**: Większość modeli językowych, w tym GPT-4, ma ograniczony kontekst - mogą przetwarzać tylko określoną liczbę tokenów na raz. Dzięki podziałowi możemy przetwarzać dużo większe dokumenty.

2. **Precyzja wyszukiwania**: Mniejsze fragmenty pozwalają na bardziej precyzyjne wyszukiwanie informacji. Zamiast wyszukiwać całe dokumenty, system może znaleźć konkretne fragmenty zawierające poszukiwane informacje.

3. **Efektywność obliczeniowa**: Przetwarzanie mniejszych fragmentów tekstu jest zazwyczaj szybsze i wymaga mniej zasobów.

4. **Lepsza wektoryzacja**: Reprezentacje wektorowe krótszych fragmentów tekstu są często bardziej znaczące i spójne semantycznie.

### Jak działa SentenceSplitter?

Klasa `SentenceSplitter` jest specjalistycznym narzędziem, które dzieli tekst na fragmenty, starając się zachować granice zdań. W przeciwieństwie do prostych podziałów na podstawie liczby słów czy znaków, ten dzielnik stara się nie przerywać zdań w połowie, co pomaga zachować spójność semantyczną fragmentów.

Parametry, które zostały przekazane do `SentenceSplitter`:

- `chunk_size = CFG.chunk_size` - określa docelowy rozmiar każdego fragmentu. Z wcześniejszego kodu wiemy, że `CFG.chunk_size = 1024`, więc każdy fragment będzie miał około 1024 tokenów/znaków.

- `chunk_overlap = CFG.chunk_overlap` - określa, ile tekstu powinno się nakładać między sąsiednimi fragmentami. Z wcześniejszego kodu wiemy, że `CFG.chunk_overlap = 20`, więc sąsiednie fragmenty będą miały wspólne około 20 tokenów/znaków.

### Czym są "nodes"?

Wynik operacji `get_nodes_from_documents` to lista obiektów typu `Node` (lub dokładniej, prawdopodobnie jakiegoś podtypu jak `TextNode`). Node to podstawowa jednostka danych w wielu systemach przetwarzania tekstu, w tym w LlamaIndex. Każdy node zazwyczaj zawiera:

- **Treść**: właściwy tekst fragmentu
- **Metadane**: informacje o źródle, pozycji w oryginalnym dokumencie, itp.
- **Identyfikator**: unikalny identyfikator node'a
- **Relacje**: potencjalne powiązania z innymi node'ami (choć na tym etapie prawdopodobnie nie są jeszcze ustalone)

### Znaczenie nakładania się fragmentów

Parametr `chunk_overlap` jest szczególnie ważny. Nakładanie się fragmentów pomaga w:

1. **Zachowaniu kontekstu**: Informacje na granicy fragmentów nie będą tracone, ponieważ pojawią się w obu fragmentach.

2. **Poprawie wyszukiwania**: Zapytania odwołujące się do informacji rozproszonych między fragmentami mogą znaleźć odpowiedni fragment dzięki nakładaniu się.

3. **Lepszej spójności**: Jeśli fragmenty będą później wykorzystywane do generowania tekstu, nakładanie się pomaga w płynnym łączeniu informacji.

Wartość 20 dla nakładania się jest stosunkowo mała (około 2% rozmiaru fragmentu), co sugeruje, że twórcy kodu oczekują, że większość istotnych informacji będzie zawarta w pojedynczych fragmentach.

### W szerszym kontekście

Ten krok dzielenia dokumentów jest zazwyczaj częścią większego procesu w systemie RAG:

1. **Pobranie dokumentów** (co widzieliśmy wcześniej)
2. **Podział dokumentów na fragmenty** (obecny krok)
3. **Wektoryzacja fragmentów** (prawdopodobnie kolejny krok)
4. **Indeksowanie wektorów** dla efektywnego wyszukiwania
5. **Wyszukiwanie właściwych fragmentów** w odpowiedzi na zapytania
6. **Generowanie odpowiedzi** z wykorzystaniem wyszukanych fragmentów

Ten fragment kodu realizuje więc kluczowy, drugi krok w tym procesie, przygotowując dokumenty do dalszego przetwarzania w systemie RAG.

In [ ]:
KG_TRIPLET_EXTRACT_TMPL = """
-Goal-
Given a text document, identify all entities and their entity types from the text and all relationships among the identified entities.
Given the text, extract up to {max_knowledge_triplets} entity-relation triplets.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: Type of the entity
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"$$$$<entity_name>$$$$<entity_type>$$$$<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relation: relationship between source_entity and target_entity
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other

Format each relationship as ("relationship"$$$$<source_entity>$$$$<target_entity>$$$$<relation>$$$$<relationship_description>)

3. When finished, output.

-Real Data-
######################
text: {text}
######################
output:"""

Ten szablon zapytania (`KG_TRIPLET_EXTRACT_TMPL`) służy do ekstrakcji trójek wiedzy (knowledge triplets) z tekstu przy użyciu modelu językowego. Przyjrzyjmy się, jak jest zbudowany i jak działa.

Szablon ma strukturę, która prowadzi model językowy przez proces analizy tekstu w celu identyfikacji encji i relacji między nimi. Podzielmy to na części:

### Cel i struktura zapytania

Zapytanie rozpoczyna się od określenia celu: identyfikacji wszystkich encji i ich typów z tekstu oraz wszystkich relacji między tymi encjami. Model ma wyodrębnić do określonej liczby trójek encja-relacja z podanego tekstu. Liczba ta jest parametryzowana przez zmienną `{max_knowledge_triplets}`.

### Krok 1: Identyfikacja encji

W pierwszym kroku model musi zidentyfikować wszystkie encje w tekście. Dla każdej encji model powinien wyodrębnić:
- `entity_name`: Nazwa encji, pisana wielką literą
- `entity_type`: Typ encji (np. osoba, organizacja, miejsce)
- `entity_description`: Kompleksowy opis atrybutów i działań encji

Każda encja ma być sformatowana według wzorca:
```
("entity"$$$$<entity_name>$$$$<entity_type>$$$$<entity_description>)
```

Zauważmy użycie `$$$$` jako separatora między różnymi elementami encji. Ten niestandardowy separator jest prawdopodobnie wybrany, aby był bardzo mało prawdopodobny do wystąpienia w normalnym tekście, co ułatwia późniejsze przetwarzanie wyjścia.

### Krok 2: Identyfikacja relacji

W drugim kroku model ma zidentyfikować wszystkie pary encji, które są wyraźnie powiązane ze sobą. Dla każdej pary powiązanych encji, model powinien wyodrębnić:
- `source_entity`: Nazwa encji źródłowej
- `target_entity`: Nazwa encji docelowej
- `relation`: Relacja między encją źródłową a docelową
- `relationship_description`: Wyjaśnienie, dlaczego te dwie encje są powiązane

Każda relacja ma być sformatowana według wzorca:
```
("relationship"$$$$<source_entity>$$$$<target_entity>$$$$<relation>$$$$<relationship_description>)
```

### Struktura szablonu i dane wejściowe

Całość szablonu używa specjalnych znaczników, które ułatwiają modelowi językowemu zrozumienie, co ma robić:
- `-Goal-`: Określa cel zadania
- `-Steps-`: Definiuje kroki do wykonania
- `-Real Data-`: Wprowadza dane do przetworzenia

Tekst wejściowy jest umieszczony między znacznikami `######################`, co dodatkowo wyróżnia go dla modelu. Zmienna `{text}` zostanie zastąpiona rzeczywistym tekstem, który ma być analizowany.

### Praktyczne zastosowanie

Ten szablon jest używany w klasie `GraphRAGExtractor`, którą analizowaliśmy wcześniej. W tej klasie, po wywołaniu modelu językowego z tym szablonem i tekstem wejściowym, wynik jest przekazywany do funkcji `parse_fn` (domyślnie `default_parse_triplets_fn`), która przetwarza surowy tekst odpowiedzi i konwertuje go na struktury danych reprezentujące encje i relacje.

### Dlaczego taki format?

Taki format jest zaprojektowany, aby uzyskać strukturalną odpowiedź od modelu językowego, która może być łatwo przetworzona programistycznie. Używanie jednoznacznych separatorów (`$$$$`) i jasnej struktury formatowania sprawia, że późniejsze parsowanie odpowiedzi modelu jest prostsze i bardziej niezawodne.

Ten szablon jest kluczowym elementem procesu ekstrakcji wiedzy strukturalnej z nieustrukturyzowanego tekstu, co jest fundamentalnym zadaniem w budowaniu grafów wiedzy. Poprzez staranne prowadzenie modelu językowego przez proces identyfikacji encji i relacji, szablon pomaga przekształcić "płaski" tekst w bogatą, strukturalną reprezentację zawartej w nim wiedzy.

In [ ]:
entity_pattern = r'\("entity"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
relationship_pattern = r'\("relationship"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'


def parse_fn(response_str: str) -> Any:
    entities = re.findall(entity_pattern, response_str)
    relationships = re.findall(relationship_pattern, response_str)
    return entities, relationships

Ten kod definiuje funkcję `parse_fn` oraz dwa wzorce wyrażeń regularnych, które służą do wyodrębnienia informacji o encjach i relacjach z odpowiedzi modelu językowego. Przeanalizujmy jak dokładnie działa ten mechanizm.

### Wyrażenia regularne

Zacznijmy od analizy dwóch wzorców wyrażeń regularnych:

```python
entity_pattern = r'\("entity"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
relationship_pattern = r'\("relationship"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\$\$\$\$"(.+?)"\)'
```

Pierwszy wzorzec, `entity_pattern`, jest zaprojektowany do wyodrębnienia informacji o encjach z tekstu odpowiedzi. Przeanalizujmy go składnik po składniku:

- `\(` - dopasowuje nawias otwierający
- `"entity"` - dopasowuje dosłownie tekst "entity" w cudzysłowach
- `\$\$\$\$` - dopasowuje cztery znaki dolara (które służą jako separator)
- `"(.+?)"` - dopasowuje i przechwytuje cokolwiek w cudzysłowach (nazwa encji)
- `\$\$\$\$` - kolejny separator
- `"(.+?)"` - drugi przechwycony element (typ encji)
- `\$\$\$\$` - kolejny separator
- `"(.+?)"` - trzeci przechwycony element (opis encji)
- `\)` - dopasowuje nawias zamykający

Znak zapytania w `(.+?)` sprawia, że dopasowanie jest "leniwe" - obejmuje minimalną ilość tekstu potrzebną do spełnienia wzorca, co jest ważne, gdy mamy wiele encji w jednym tekście.

Drugi wzorzec, `relationship_pattern`, działa podobnie, ale ma cztery grupy przechwytujące odpowiadające czterem elementom relacji: encja źródłowa, encja docelowa, nazwa relacji i opis relacji.

### Funkcja parse_fn

Funkcja `parse_fn` przyjmuje jako argument ciąg znaków `response_str`, który jest odpowiedzią wygenerowaną przez model językowy według wcześniej omówionego szablonu `KG_TRIPLET_EXTRACT_TMPL`.

```python
def parse_fn(response_str: str) -> Any:
    entities = re.findall(entity_pattern, response_str)
    relationships = re.findall(relationship_pattern, response_str)
    return entities, relationships
```

Funkcja wykorzystuje metodę `re.findall()` z modułu `re` (Regular Expressions) do znalezienia wszystkich dopasowań do każdego wzorca:

1. `entities = re.findall(entity_pattern, response_str)` znajduje wszystkie wystąpienia wzorca encji i zapisuje je jako listę krotek, gdzie każda krotka zawiera trzy elementy: nazwę encji, typ encji i opis encji.

2. `relationships = re.findall(relationship_pattern, response_str)` znajduje wszystkie wystąpienia wzorca relacji i zapisuje je jako listę krotek, gdzie każda krotka zawiera cztery elementy: encję źródłową, encję docelową, nazwę relacji i opis relacji.

3. Funkcja zwraca krotkę zawierającą dwie listy: listę encji i listę relacji.

### Połączenie z wcześniejszym kodem

Łącząc tę funkcję z szablonem `KG_TRIPLET_EXTRACT_TMPL` i klasą `GraphRAGExtractor`, mamy kompletny mechanizm do:
1. Generowania zapytania, które prosi model językowy o wyodrębnienie encji i relacji z tekstu
2. Parsowania odpowiedzi modelu, aby uzyskać strukturalną reprezentację wyodrębnionych informacji
3. Konwersji tych informacji na węzły i relacje w grafie wiedzy

Warto zauważyć, że ten typ architektury LLM-as-a-parser (model językowy jako parser) jest coraz bardziej popularnym podejściem w przetwarzaniu języka naturalnego. Zamiast pisać skomplikowane reguły parsowania, które mogą być kruche i trudne do utrzymania, wykorzystujemy zdolności rozumienia języka naturalnego przez duże modele językowe do wyodrębnienia strukturalnych informacji.

Jednocześnie, aby zapewnić spójność i możliwość programistycznego przetwarzania, używamy jasno zdefiniowanego formatu wyjściowego (z separatorami `$$$$`) i wyrażeń regularnych do interpretacji tego wyjścia. To hybrydowe podejście łączy elastyczność modeli językowych z precyzją i determinizmem tradycyjnych metod parsowania.

In [ ]:
kg_extractor = GraphRAGExtractor(
    llm=llm,
    extract_prompt=KG_TRIPLET_EXTRACT_TMPL,
    max_paths_per_chunk=2,
    parse_fn=parse_fn,
)

Ta linia kodu inicjalizuje obiekt `GraphRAGExtractor`, który odgrywa kluczową rolę w przekształcaniu tekstu w strukturę grafu wiedzy. Przyjrzyjmy się dokładnie, co się tutaj dzieje i dlaczego jest to istotne.

### Co to jest GraphRAGExtractor?

`GraphRAGExtractor` to specjalistyczna klasa, którą analizowaliśmy wcześniej, zaprojektowana do wyodrębniania wiedzy strukturalnej (tzw. "trójek" lub "tripletów") z tekstu. Jest to komponent systemu RAG (Retrieval-Augmented Generation), który łączy wyszukiwanie informacji z generowaniem tekstu.

Wyobraź sobie, że czytasz długi artykuł i robisz notatki w formie "Kto - co robi - komu/czemu". Na przykład "Einstein - opracował - teorię względności". To właśnie robi `GraphRAGExtractor` - ale automatycznie, z pomocą modelu językowego.

### Parametry inicjalizacji

Ten obiekt jest inicjalizowany z czterema konkretnymi parametrami:

1. **`llm=llm`**: Przekazujemy wcześniej utworzony obiekt `llm`, który jest interfejsem do modelu GPT-4. Model ten będzie odpowiedzialny za "rozumienie" tekstu i identyfikację encji i relacji. To tak, jakbyśmy zatrudnili bardzo inteligentnego asystenta do czytania tekstu i wyciągania z niego najważniejszych informacji.

2. **`extract_prompt=KG_TRIPLET_EXTRACT_TMPL`**: Przekazujemy szablon zapytania, który definiuje, jak dokładnie chcemy, aby model językowy wyodrębniał informacje. Ten szablon, który analizowaliśmy wcześniej, zawiera szczegółowe instrukcje dla modelu, co ma zrobić z tekstem - zidentyfikować encje, określić ich typy, opisać je i zidentyfikować relacje między nimi.

3. **`max_paths_per_chunk=2`**: Ten parametr ogranicza liczbę trójek (ścieżek w grafie), które będą wyodrębniane z każdego fragmentu tekstu, do dwóch. Jest to dość niska wartość, co sugeruje, że twórcy kodu chcą skupić się na najbardziej istotnych relacjach lub może to być ustawienie do celów testowych.

4. **`parse_fn=parse_fn`**: Przekazujemy funkcję parsującą, którą właśnie analizowaliśmy. Ta funkcja będzie odpowiedzialna za przekształcenie odpowiedzi modelu językowego (która jest zwykłym tekstem) w struktury danych, które mogą być łatwo przetwarzane przez program - listy encji i relacji.

### Jak to działa w praktyce?

Wyobraźmy sobie, że mamy tekst opisujący teorię względności. Po uruchomieniu `GraphRAGExtractor` na tym tekście, proces będzie wyglądał następująco:

1. Tekst jest przekazywany do modelu językowego wraz z szablonem zapytania `KG_TRIPLET_EXTRACT_TMPL`.

2. Model językowy analizuje tekst i generuje odpowiedź sformatowaną zgodnie z instrukcjami z szablonu, na przykład:
   ```
   ("entity"$$$$"Albert Einstein"$$$$"Naukowiec"$$$$"Fizyk teoretyczny, który zrewolucjonizował fizykę w XX wieku...")
   ("entity"$$$$"Teoria względności"$$$$"Teoria naukowa"$$$$"Teoria fizyczna opisująca przestrzeń, czas i grawitację...")
   ("relationship"$$$$"Albert Einstein"$$$$"Teoria względności"$$$$"opracował"$$$$"Einstein opublikował ogólną teorię względności w 1915 roku...")
   ```

3. Funkcja `parse_fn` przetwarza tę odpowiedź, wyodrębniając encje i relacje.

4. Te encje i relacje są następnie przekształcane w węzły i krawędzie grafu wiedzy, które mogą być przechowywane, wyszukiwane i wykorzystywane do generowania odpowiedzi na pytania.

### Dlaczego to jest ważne?

Ten mechanizm ekstrakcji wiedzy jest fundamentalny dla systemu RAG, ponieważ:

1. **Transformuje niestrukturyzowany tekst w strukturę grafową**: Grafy wiedzy są potężnymi reprezentacjami danych, które mogą wyrażać złożone relacje między encjami. Przekształcenie tekstu w graf umożliwia bardziej zaawansowane wyszukiwanie i wnioskowanie.

2. **Wykorzystuje zdolności rozumienia języka naturalnego LLM**: Zamiast polegać na sztywnych regułach ekstrakcji informacji, system wykorzystuje zaawansowane zdolności modeli językowych do zrozumienia kontekstu i semantyki.

3. **Umożliwia generowanie odpowiedzi opartych na faktach**: Dzięki grafowi wiedzy, system może generować odpowiedzi, które są bezpośrednio powiązane z faktami z oryginalnych dokumentów, co zwiększa wiarygodność i precyzję.

4. **Ułatwia eksplorację i odkrywanie wiedzy**: Struktura grafowa ułatwia odkrywanie nowych powiązań między fragmentami informacji, które mogły nie być oczywiste w oryginalnym tekście.

W kontekście całego systemu RAG, ten ekstraktor jest mostkiem między surowym tekstem a bogatą, strukturalną reprezentacją wiedzy, która może być efektywnie wykorzystywana do odpowiadania na pytania i generowania nowych treści.

In [ ]:
index = PropertyGraphIndex(
    nodes=nodes,
    property_graph_store=GraphRAGStore(),
    kg_extractors=[kg_extractor],
    show_progress=True,
)

Generating embeddings: 100%|██████████| 4/4 [00:01<00:00,  3.73it/s]


Ten fragment kodu tworzy i inicjalizuje indeks grafowy (`PropertyGraphIndex`), który jest sercem systemu RAG (Retrieval-Augmented Generation) opartego na grafie wiedzy. Spróbujmy dogłębnie zrozumieć, co się tutaj dzieje i dlaczego ma to znaczenie.

### Co to jest PropertyGraphIndex?

`PropertyGraphIndex` to specjalistyczna struktura danych, która łączy fragmenty tekstu (nodes) z grafem wiedzy (property_graph). Możemy o tym myśleć jako o zaawansowanej książce z indeksem, gdzie każde hasło (fragment tekstu) jest powiązane z siecią pojęć i relacji (graf wiedzy).

### Parametry inicjalizacji

Indeks jest tworzony z czterema głównymi parametrami:

1. **`nodes=nodes`** - Przekazujemy listę węzłów (fragmentów tekstu), które zostały wcześniej utworzone przez `SentenceSplitter`. Te węzły zawierają właściwy tekst dokumentów, podzielony na mniejsze, łatwiejsze do przetwarzania fragmenty. To są "surowe dane", z których będziemy wydobywać wiedzę strukturalną.

2. **`property_graph_store=GraphRAGStore()`** - Tworzymy nową instancję `GraphRAGStore`, specjalizowanego magazynu do przechowywania grafu wiedzy. Ten magazyn będzie nie tylko przechowywał węzły i relacje grafu, ale także, jak widzieliśmy w analizie klasy `GraphRAGStore`, będzie organizował je w społeczności (communities) i generował dla nich podsumowania.

3. **`kg_extractors=[kg_extractor]`** - Przekazujemy listę ekstraktorów wiedzy, w tym przypadku zawierającą tylko jeden element - nasz wcześniej skonfigurowany `kg_extractor`. Te ekstraktory są odpowiedzialne za przekształcenie tekstu z węzłów w strukturę grafową. W praktyce można by użyć wielu różnych ekstraktorów, każdy specjalizujący się w innym typie wiedzy lub relacji.

4. **`show_progress=True`** - Ten parametr włącza wyświetlanie postępu podczas budowania indeksu. To przydatna funkcja, szczególnie przy dużych zbiorach danych, gdzie proces może trwać dłużej.

### Co się dzieje podczas inicjalizacji?

Kiedy tworzymy `PropertyGraphIndex`, dzieje się kilka ważnych rzeczy:

1. **Ekstrakcja wiedzy**: Każdy węzeł (fragment tekstu) jest przetwarzany przez ekstraktor wiedzy (`kg_extractor`). Ekstraktor wykorzystuje model językowy, aby zidentyfikować encje i relacje w tekście, jak omówiliśmy wcześniej.

2. **Budowanie grafu**: Zidentyfikowane encje stają się węzłami grafu wiedzy, a relacje między nimi stają się krawędziami. Ten graf jest przechowywany w `property_graph_store`.

3. **Organizowanie społeczności**: Jeśli `property_graph_store` wykorzystuje funkcje organizacji społeczności (co widzieliśmy w klasie `GraphRAGStore`), graf zostanie podzielony na mniejsze, spójne tematycznie grupy.

4. **Generowanie podsumowań**: Dla każdej społeczności w grafie może zostać wygenerowane podsumowanie, które będzie później wykorzystywane do odpowiadania na pytania.

### Znaczenie tego komponentu w systemie RAG

Ten indeks grafowy jest kluczowym elementem systemu RAG z kilku powodów:

1. **Łączy tekst ze strukturą**: Indeks tworzy powiązania między oryginalnym tekstem a strukturalną reprezentacją wiedzy w formie grafu. Dzięki temu system może odpowiadać na pytania zarówno na podstawie dosłownego tekstu, jak i na podstawie relacji i faktów wyekstrahowanych z tekstu.

2. **Umożliwia zaawansowane wyszukiwanie**: Dzięki strukturze grafowej, system może przeprowadzać zaawansowane wyszukiwania, które uwzględniają nie tylko słowa kluczowe, ale także relacje między encjami. Na przykład, zamiast szukać wszystkich fragmentów zawierających słowa "Einstein" i "względność", system może bezpośrednio znaleźć relację "Einstein opracował teorię względności".

3. **Wspiera generowanie odpowiedzi opartych na faktach**: Mając dostęp do strukturalnej wiedzy w formie grafu, system może generować odpowiedzi, które są bezpośrednio powiązane z faktami z dokumentów, co zwiększa wiarygodność i precyzję.

4. **Ułatwia odkrywanie nowych powiązań**: Struktura grafowa może ujawnić powiązania między fragmentami informacji, które mogły nie być oczywiste w oryginalnym tekście, co może prowadzić do nowych spostrzeżeń i odkryć.

### W praktycznym zastosowaniu

Wyobraźmy sobie, że mamy kolekcję artykułów naukowych o zmianach klimatycznych. Po przetworzeniu przez ten system:

1. Każdy artykuł zostanie podzielony na mniejsze fragmenty (nodes).
2. Z każdego fragmentu zostaną wyekstrahowane encje (np. "dwutlenek węgla", "efekt cieplarniany", "poziom mórz") i relacje między nimi (np. "dwutlenek węgla zwiększa efekt cieplarniany", "efekt cieplarniany podnosi poziom mórz").
3. Te encje i relacje utworzą graf wiedzy, który będzie przechowywany w `property_graph_store`.
4. Graf zostanie podzielony na społeczności tematyczne (np. "gazy cieplarniane", "skutki dla oceanów", "działania mitygacyjne").
5. Dla każdej społeczności zostanie wygenerowane podsumowanie.

Teraz, gdy użytkownik zada pytanie, na przykład "Jaki wpływ ma dwutlenek węgla na poziom mórz?", system może:
1. Zidentyfikować odpowiednie społeczności w grafie (np. "gazy cieplarniane" i "skutki dla oceanów").
2. Znaleźć ścieżkę w grafie łączącą "dwutlenek węgla" z "poziomem mórz".
3. Wygenerować odpowiedź opartą na podsumowaniach społeczności i znalezionej ścieżce.

Ten zaawansowany proces umożliwia generowanie odpowiedzi, które są zarówno precyzyjne (oparte na faktach z dokumentów), jak i kompleksowe (uwzględniające złożone relacje między faktami).

In [ ]:

index.property_graph_store.build_communities()

/usr/local/lib/python3.11/dist-packages/graspologic/partition/leiden.py:607: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


Ta linia kodu uruchamia proces budowania społeczności (communities) w grafie wiedzy, który został wcześniej utworzony przez `PropertyGraphIndex`. To bardzo ważny krok w przygotowaniu grafu wiedzy do efektywnego wyszukiwania i generowania odpowiedzi.

Przypomnijmy, co dokładnie dzieje się podczas wywołania metody `build_communities()` w `GraphRAGStore` (na podstawie wcześniejszej analizy tej klasy):

1. **Konwersja grafu wewnętrznego na graf NetworkX**:
   Najpierw wewnętrzna reprezentacja grafu jest konwertowana na format biblioteki NetworkX, która oferuje zaawansowane algorytmy do analizy grafów. W tym procesie każdy węzeł i każda relacja z wewnętrznego grafu są przekształcane w odpowiednie elementy grafu NetworkX.

2. **Identyfikacja hierarchicznych klastrów za pomocą algorytmu Leiden**:
   Następnie na grafie NetworkX jest uruchamiany algorytm Leiden, specjalistyczny algorytm do wykrywania społeczności w grafach. Algorytm ten identyfikuje grupy węzłów, które są silnie połączone wewnętrznie, a słabiej z resztą grafu. Parametr `max_cluster_size` (który w przypadku klasy `GraphRAGStore` ma wartość 5) określa maksymalny rozmiar każdej społeczności, co prowadzi do podziału grafu na mniejsze, bardziej spójne tematycznie grupy.

3. **Zbieranie informacji o społecznościach**:
   Dla każdej zidentyfikowanej społeczności, metoda zbiera szczegółowe informacje o relacjach między węzłami należącymi do tej społeczności. Te informacje są formatowane jako tekstowe opisy w stylu "węzeł1 -> węzeł2 -> relacja -> opis_relacji".

4. **Generowanie podsumowań dla społeczności**:
   Na końcu, dla każdej społeczności jest generowane podsumowanie za pomocą modelu językowego. Model otrzymuje tekstowe opisy relacji w społeczności i tworzy zwięzłe podsumowanie, które podkreśla kluczowe aspekty tych relacji.

Wynik całego tego procesu to słownik `community_summary`, gdzie kluczami są identyfikatory społeczności, a wartościami są ich podsumowania. Te podsumowania będą później używane przez `GraphRAGQueryEngine` do generowania odpowiedzi na zapytania użytkownika.

### Dlaczego budowanie społeczności jest ważne?

Budowanie społeczności w grafie wiedzy ma kilka istotnych zalet:

1. **Organizacja wiedzy**: Podział grafu na mniejsze, spójne tematycznie grupy pomaga w organizacji wiedzy, ułatwiając późniejsze wyszukiwanie i przetwarzanie.

2. **Poprawa efektywności**: Zamiast przetwarzać cały graf (który może być bardzo duży i złożony) dla każdego zapytania, system może najpierw zidentyfikować odpowiednie społeczności, a następnie skupić się na nich.

3. **Lepsza interpretacja**: Podsumowania społeczności oferują wysokopoziomowy widok na wiedzę zawartą w grafie, co może być łatwiejsze do interpretacji zarówno dla systemu, jak i dla użytkownika końcowego.

4. **Odkrywanie powiązań**: Grupowanie węzłów w społeczności może ujawnić powiązania między fragmentami informacji, które mogły nie być oczywiste w oryginalnym tekście.

### W praktycznym zastosowaniu

Wyobraźmy sobie, że nasz graf wiedzy zawiera informacje o różnych aspektach zmian klimatycznych. Po zbudowaniu społeczności, możemy otrzymać grupy jak:

- Społeczność 1: Węzły i relacje dotyczące gazów cieplarnianych (CO2, metan, wpływ na atmosferę)
- Społeczność 2: Węzły i relacje dotyczące skutków dla oceanów (poziom mórz, zakwaszenie, wpływ na ekosystemy morskie)
- Społeczność 3: Węzły i relacje dotyczące działań mitygacyjnych (energia odnawialna, redukcja emisji, polityka klimatyczna)

Dla każdej z tych społeczności zostanie wygenerowane podsumowanie, które podkreśla kluczowe aspekty relacji w tej grupie. Na przykład, podsumowanie Społeczności 1 mogłoby brzmieć:

"Ta społeczność koncentruje się na gazach cieplarnianych, ze szczególnym uwzględnieniem dwutlenku węgla i metanu. Dwutlenek węgla jest głównym gazem cieplarnianym emitowanym przez działalność człowieka, szczególnie przez spalanie paliw kopalnych. Metan, choć emitowany w mniejszych ilościach, ma silniejszy efekt cieplarniany w krótkim okresie. Oba te gazy przyczyniają się do ocieplenia atmosfery poprzez zatrzymywanie ciepła, co prowadzi do globalnego wzrostu temperatury."

Takie podsumowania będą później używane przez `GraphRAGQueryEngine` do generowania odpowiedzi na pytania użytkownika, zapewniając, że odpowiedzi są zarówno precyzyjne (oparte na faktach z grafu), jak i kontekstowe (uwzględniające szerszy kontekst społeczności).

Wywołanie `index.property_graph_store.build_communities()` jest zatem kluczowym krokiem w przygotowaniu systemu RAG do efektywnego odpowiadania na pytania użytkownika, transformując surowy graf wiedzy w zorganizowaną strukturę, która może być skutecznie wykorzystana do generowania wartościowych odpowiedzi.

In [ ]:
query_engine = GraphRAGQueryEngine(
    graph_store=index.property_graph_store, llm=llm
)

Ta linia kodu tworzy silnik zapytań `GraphRAGQueryEngine`, który będzie odpowiedzialny za przetwarzanie pytań użytkownika i generowanie odpowiedzi na podstawie wiedzy zgromadzonej w grafie. To ostatni, kluczowy element całego systemu RAG (Retrieval-Augmented Generation).

## Jak działa GraphRAGQueryEngine?

Wyobraź sobie, że masz bibliotekę z tysiącami książek (dokumenty), ale zamiast przeszukiwać każdą z nich od początku do końca, masz inteligentnego bibliotekarza (GraphRAGQueryEngine), który:

1. Zna wszystkie główne tematy w bibliotece (społeczności w grafie)
2. Dla każdego tematu ma przygotowane krótkie podsumowanie najważniejszych informacji
3. Potrafi łączyć informacje z różnych tematów, aby odpowiedzieć na złożone pytania

Kiedy zadajesz pytanie, bibliotekarz:
- Identyfikuje, których tematów dotyczy twoje pytanie
- Przegląda swoje podsumowania tych tematów
- Formułuje odpowiedź, łącząc najistotniejsze informacje

## Co jest przekazywane do GraphRAGQueryEngine?

1. **`graph_store=index.property_graph_store`** - Przekazujemy magazyn grafu, który zawiera:
   - Węzły grafu (encje wyodrębnione z tekstu)
   - Relacje między tymi węzłami
   - Społeczności (grupy powiązanych węzłów i relacji)
   - Podsumowania tych społeczności

   To jest "mapa wiedzy" wyekstrahowana z oryginalnych dokumentów, zorganizowana w sposób, który ułatwia wyszukiwanie i łączenie informacji.

2. **`llm=llm`** - Przekazujemy model językowy (w tym przypadku GPT-4), który będzie używany do:
   - Analizowania pytania użytkownika
   - Generowania odpowiedzi na podstawie podsumowań społeczności
   - Łączenia częściowych odpowiedzi w spójną całość

## Proces generowania odpowiedzi

Gdy użytkownik zada pytanie, proces wygląda następująco:

1. **Pobranie podsumowań społeczności**: Silnik zapytań pobiera wszystkie podsumowania społeczności z grafu wiedzy.

2. **Generowanie częściowych odpowiedzi**: Dla każdego podsumowania społeczności, model językowy generuje częściową odpowiedź na pytanie użytkownika. Te częściowe odpowiedzi są tworzone na podstawie wiedzy zawartej w poszczególnych społecznościach.

   Na przykład, jeśli pytanie dotyczy wpływu dwutlenku węgla na poziom mórz, a mamy społeczności o gazach cieplarnianych i o oceanach, model wygeneruje dwie częściowe odpowiedzi - jedną na podstawie wiedzy o gazach cieplarnianych, drugą na podstawie wiedzy o oceanach.

3. **Agregacja odpowiedzi**: Wszystkie częściowe odpowiedzi są łączone w jedną, spójną odpowiedź końcową. Model językowy nie tylko łączy te odpowiedzi, ale także usuwa powtórzenia, rozwiązuje ewentualne sprzeczności i zapewnia płynne przejścia między różnymi aspektami odpowiedzi.

## Dlaczego to podejście jest skuteczne?

Ta architektura ma kilka istotnych zalet:

1. **Lepsze zrozumienie kontekstu**: Dzięki podziałowi wiedzy na społeczności, system lepiej rozumie kontekst każdej informacji i relacje między różnymi fragmentami wiedzy.

2. **Większa precyzja**: Odpowiedzi są oparte na faktach wyekstrahowanych z dokumentów i zorganizowanych w strukturę grafową, co zwiększa ich precyzję i wiarygodność.

3. **Kompleksowość**: System może łączyć informacje z różnych społeczności, tworząc odpowiedzi, które uwzględniają różne aspekty pytania.

4. **Adaptacyjność**: Ta architektura jest elastyczna - może obsługiwać różne typy pytań, od prostych faktograficznych po złożone, wymagające łączenia informacji z wielu źródeł.

## Zastosowanie w praktyce

Wyobraź sobie, że zadajesz pytanie: "Jakie są powiązania między poziomem CO2 a ekonomią?"

System:
1. Identyfikuje, że to pytanie może dotyczyć kilku społeczności w grafie: "gazy cieplarniane", "gospodarka", "polityka klimatyczna"
2. Generuje częściowe odpowiedzi na podstawie podsumowań tych społeczności:
   - Z "gazów cieplarnianych": informacje o emisjach CO2 z przemysłu
   - Z "gospodarki": wpływ zmian klimatycznych na różne sektory gospodarki
   - Z "polityki klimatycznej": ekonomiczne aspekty redukcji emisji
3. Łączy te częściowe odpowiedzi w jedną kompleksową, która uwzględnia wszystkie istotne aspekty pytania

Ta linia kodu jest więc zwieńczeniem całego procesu budowy systemu RAG - tworzy "interfejs" między użytkownikiem a zgromadzoną i zorganizowaną wiedzą, umożliwiając zadawanie pytań i otrzymywanie inteligentnych, opartych na faktach odpowiedzi.

In [ ]:
response = query_engine.query( "What are the main news discussed in the document?")
display(Markdown(f"{response.response}"))

The document discusses various news topics including Chevron's stock NYSE:CVX, Stellantis' plans to close the Belvidere Assembly Plant affecting Jeep Cherokee production, and FirstEnergy's earnings results. It also covers a protest by Sinn Féin TD John Brady against Minister for Housing Darragh O’Brien, the development of the MetaHuman Animator tool by Epic, and the European Commission's decision to ban TikTok Inc. due to data-collection concerns. Other news includes Manchester United's withdrawal from signing Harry Kane, Jude Bellingham's transfer to Real Madrid, the tease of the Vivo X90s smartphone, and the negotiation for a contract extension between Maliek Collins and the Houston Texans. The document also discusses the acquisition of The Hollies' recording catalog by BMG, various partnerships and collaborations in the music and tech industries, and the partnership between Supplier.io and Hyatt Hotels. It mentions the creation of "Star Ocean: The Second Story R" by Square Enix, a drop in uninsured deposits by major banks, and an unusually high charge reported by a Paytm FASTag user. Other topics include JetBlue's aircraft "A Defining MoMint", Coinbase Global Inc.'s repurchase of convertible senior notes, Vincent Kompany's role in Manchester City, the closure of Protonn, and the employment of Thomas Christl by Morgan Stanley. The document also covers the launch of the Redmi 12C smartphone and the Hyundai Exter in India, the upgrade of Allegiant Travel's rating by Deutsche Bank, the impacts of the COVID-19 pandemic on Delta Air Lines and Southwest Airlines, potential Champions League matchups, and various NFL player updates. It also mentions Arsenal's unsuccessful bid for Rice, Chelsea's interest in André Onana, the uncertainty of David de Gea's contract with Manchester United, Bank of America's investment in Papaya, and the competitive relationship between UnitedHealth Group Inc. and Humana Inc. Finally, it discusses the Houston Texans signing Maliek Collins, Shereen Zahawi's role at MDA Ltd., and MDA Ltd.'s participation in the Jefferies Virtual Space Summit.

In [ ]:
response = query_engine.query("What are news related to financial sector?")
display(Markdown(f"{response.response}"))

The financial sector news includes Chevron's stock performance on the NYSE and Stellantis's plan to close the Belvidere Assembly Plant, affecting the production of the Jeep Cherokee. Flipkart is collaborating with Hang to introduce a loyalty program, FireDrops 2.0, and Scapia has partnered with Federal Bank to launch a global credit card. Hyatt Hotels has been honored with the 2023 Top Supply Chain Projects award. Major banks are reporting their changes in uninsured deposits to S&P Global Inc. A user named Emvi experienced an unusually high charge through Paytm FASTag, indicating a potential system error. Coinbase Global Inc. repurchased $64.5 million worth of convertible senior notes but faced regulatory challenges from the U.S. Securities and Exchange Commission. Morgan Stanley hired Thomas Christl to co-head its European consumer and retail client coverage. Deutsche Bank upgraded Allegiant Travel's rating from 'Hold' to 'Buy'. Delta Air Lines and Southwest Airlines were significantly affected by the COVID-19 pandemic, with both suspending their dividend payouts. Bank of America invested in Papaya through Chingona Ventures, and Matt Pincus led a $15 million pre-growth round for Soundtrack Your Brand. UnitedHealth Group Inc. has a significant influence over the Dow Jones Industrial Average due to its high-priced stock. Lastly, Shereen Zahawi holds the position of Senior Director of Investor Relations at MDA Ltd., which will participate in the Jefferies Virtual Space Summit.